In [ ]:
from flask import Flask, jsonify, request, render_template
import sqlite3
from pathlib import Path

app = Flask(__name__)

def get_db_path():
    """Get the path to the boilest.db database"""
    return Path.cwd() / 'boilest.db'

@app.route('/', methods=['GET'])
def index():
    """Serve the main UI page"""
    print("[UI] Serving main page")
    return render_template('index.html')

@app.route('/api/encode/largest', methods=['GET'])
def get_largest_encode():
    """
    Query the encode table and return one row sorted by before_file_size DESC limit 1
    """
    print("\n" + "="*80)
    print("[REQUEST] GET /api/encode/largest")
    print("="*80)
    
    try:
        db_path = get_db_path()
        print(f"[DB] Database path: {db_path}")
        print(f"[DB] Database exists: {db_path.exists()}")
        
        conn = sqlite3.connect(db_path)
        conn.row_factory = sqlite3.Row  # This allows accessing columns by name
        cur = conn.cursor()
        print("[DB] Connected to database successfully")
        
        # Query the encode table sorted by before_file_size descending, limit 1
        query = """
            SELECT 
                guid,
                directory_path,
                input_file_name,
                output_file_name,
                before_file_size,
                decision,
                ffmpeg_string,
                date_added
            FROM encode
            ORDER BY before_file_size DESC
            LIMIT 1
        """
        print("[QUERY] Executing query to fetch largest encode by before_file_size...")
        cur.execute(query)
        
        row = cur.fetchone()
        print(f"[QUERY] Query executed successfully")
        print(f"[RESULT] Row found: {row is not None}")
        
        conn.close()
        print("[DB] Connection closed")
        
        if row:
            # Convert row to dictionary
            result = dict(row)
            print(f"[RESULT] Returning encode record with before_file_size: {result['before_file_size']} bytes")
            print("="*80 + "\n")
            return jsonify({
                'success': True,
                'data': result
            }), 200
        else:
            print("[RESULT] No records found in encode table")
            print("="*80 + "\n")
            return jsonify({
                'success': True,
                'data': None,
                'message': 'No records found in encode table'
            }), 200
            
    except Exception as e:
        print(f"[ERROR] Exception occurred: {type(e).__name__}")
        print(f"[ERROR] Error message: {str(e)}")
        print("="*80 + "\n")
        return jsonify({
            'success': False,
            'error': str(e)
        }), 500

@app.route('/api/encoded', methods=['POST'])
def create_encoded():
    """
    Write a new record to the encoded table
    Expects JSON body with: guid, after_file_size
    date_added is automatically set by the database
    """
    print("\n" + "="*80)
    print("[REQUEST] POST /api/encoded")
    print("="*80)
    
    try:
        # Get JSON data from request
        data = request.get_json()
        print(f"[REQUEST] Received data: {data}")
        
        # Validate required fields
        if not data:
            print("[ERROR] No JSON data provided")
            return jsonify({
                'success': False,
                'error': 'No JSON data provided'
            }), 400
        
        guid = data.get('guid')
        after_file_size = data.get('after_file_size')
        
        if not guid:
            print("[ERROR] Missing required field: guid")
            return jsonify({
                'success': False,
                'error': 'Missing required field: guid'
            }), 400
        
        if after_file_size is None:
            print("[ERROR] Missing required field: after_file_size")
            return jsonify({
                'success': False,
                'error': 'Missing required field: after_file_size'
            }), 400
        
        # Connect to database
        db_path = get_db_path()
        print(f"[DB] Database path: {db_path}")
        conn = sqlite3.connect(db_path)
        cur = conn.cursor()
        print("[DB] Connected to database successfully")
        
        # Insert into encoded table
        # date_added will be set automatically by DEFAULT CURRENT_TIMESTAMP
        query = """
            INSERT INTO encoded (guid, after_file_size)
            VALUES (?, ?)
        """
        print(f"[INSERT] Inserting record - guid: {guid}, after_file_size: {after_file_size}")
        cur.execute(query, (guid, after_file_size))
        conn.commit()
        
        # Get the inserted record to return it
        cur.execute("SELECT guid, after_file_size, date_added FROM encoded WHERE guid = ?", (guid,))
        row = cur.fetchone()
        
        conn.close()
        print("[DB] Record inserted successfully")
        print("[DB] Connection closed")
        print("="*80 + "\n")
        
        if row:
            result = {
                'guid': row[0],
                'after_file_size': row[1],
                'date_added': row[2]
            }
            return jsonify({
                'success': True,
                'message': 'Record created successfully',
                'data': result
            }), 201
        else:
            return jsonify({
                'success': False,
                'error': 'Record created but could not be retrieved'
            }), 500
            
    except sqlite3.IntegrityError as e:
        print(f"[ERROR] Integrity error: {str(e)}")
        print("="*80 + "\n")
        return jsonify({
            'success': False,
            'error': f'Database integrity error: {str(e)} (guid may already exist)'
        }), 409
    except Exception as e:
        print(f"[ERROR] Exception occurred: {type(e).__name__}")
        print(f"[ERROR] Error message: {str(e)}")
        print("="*80 + "\n")
        return jsonify({
            'success': False,
            'error': str(e)
        }), 500

@app.route('/health', methods=['GET'])
def health():
    """Health check endpoint"""
    print("[HEALTH CHECK] Endpoint called")
    return jsonify({'status': 'healthy'}), 200

if __name__ == '__main__':
    print("\n" + "="*80)
    print("Starting Encode API Server...")
    print("="*80)
    print("Available endpoints:")
    print("  - GET  /                    (Main UI page)")
    print("  - GET  /api/encode/largest  (Get largest encode by file size)")
    print("  - POST /api/encoded         (Create new encoded record)")
    print("  - GET  /health              (Health check)")
    print("="*80 + "\n")
    app.run(debug=False, port=5000)


Starting Encode API Server...
Available endpoints:
  - GET  /                    (Main UI page)
  - GET  /api/encode/largest  (Get largest encode by file size)
  - POST /api/encoded         (Create new encoded record)
  - GET  /health              (Health check)

 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
127.0.0.1 - - [11/Jan/2026 12:31:19] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [11/Jan/2026 12:31:19] "GET /favicon.ico HTTP/1.1" 404 -


[UI] Serving main page


127.0.0.1 - - [11/Jan/2026 12:31:25] "GET / HTTP/1.1" 200 -


[UI] Serving main page


...

[UI] Serving main page


127.0.0.1 - - [11/Jan/2026 12:31:27] "GET / HTTP/1.1" 200 -


[UI] Serving main page


127.0.0.1 - - [11/Jan/2026 12:31:28] "GET / HTTP/1.1" 200 -


[UI] Serving main page
